In [16]:
import os
import fitz 
import base64
from dotenv import load_dotenv

from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import default_transforms, apply_transforms

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from ragas.embeddings import LangchainEmbeddingsWrapper


In [2]:
load_dotenv()

True

In [3]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [17]:
embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\rayvi\AppData\Local\Temp\ipykernel_18104\2304759577.py:1: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [13]:
ragas_llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0, 
    rate_limiter=rate_limiter,
    model_kwargs={"response_format": {"type": "json_object"}}
    )

In [6]:
pdf_path = "pdfs/supplier_guide_detailed.pdf"

In [45]:
def generate_kg(pdf_path):
    kg = KnowledgeGraph()

    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]

        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_bytes = pix.tobytes("png")
        img_base64 = base64.b64encode(img_bytes).decode("utf-8")

        summary_prompt =  '''
            Describe this PDF page in detail for a RAG test set.
            Note all charts, tables, images, specific values and key text segments.
            If you see a table, output it as a markdown table.
            If you see a chart, list the data points as a bulleted list. 
            Use headers like ### Visual Data so the extractor picks them up. 
        '''

        message = HumanMessage(
            content=[
                {"type": "text", "text": summary_prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_base64}"},
                },
            ]
        )

        visual_summary = llm.invoke([message]).content

        node = Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": visual_summary,
                "image_base64": f"data:image/png;base64,{img_base64}",
                "page_number": page_num + 1
            }
        )
        kg.nodes.append(node)

    return kg


In [46]:
def generate_testset(kg, testset_size = 10):
    documents = [
        Document(
        page_content=node.properties["page_content"], 
        metadata={"page_number": node.properties["page_number"]}
        ) 
        for node in kg.nodes

    ]

    generator = TestsetGenerator(
        llm = ragas_llm,
        embedding_model = embeddings,
        knowledge_graph = kg
    )

    transforms = default_transforms(
        documents = documents,
        llm = ragas_llm,
        embedding_model = embeddings
    )

    apply_transforms(kg, transforms)

    testset = generator.generate(testset_size)
    return testset

In [47]:
kg = generate_kg(pdf_path)

In [85]:
testset = generate_testset(kg)

Applying SummaryExtractor:   4%|▍         | 1/24 [00:03<01:26,  3.77s/it]Property 'summary' already exists in node '7968e9'. Skipping!
Property 'summary' already exists in node '4d9aa0'. Skipping!
Property 'summary' already exists in node '68a061'. Skipping!
Property 'summary' already exists in node 'a6ece3'. Skipping!
Applying SummaryExtractor:  12%|█▎        | 3/24 [00:09<00:57,  2.74s/it]Property 'summary' already exists in node '5fd0f5'. Skipping!
Property 'summary' already exists in node 'b9b517'. Skipping!
Property 'summary' already exists in node '7ea0b1'. Skipping!
Property 'summary' already exists in node '1dd458'. Skipping!
Property 'summary' already exists in node 'a57e2d'. Skipping!
Property 'summary' already exists in node '12de0a'. Skipping!
Property 'summary' already exists in node '7b1f74'. Skipping!
Property 'summary' already exists in node 'bd67f4'. Skipping!
Property 'summary' already exists in node 'c1cb73'. Skipping!
Property 'summary' already exists in node '820a6

In [49]:
df = testset.to_pandas()

In [50]:
df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the purpose of the Government Procurem...,[### Key Text Segments\n- Title: **Government ...,The Government Procurement Guide for Suppliers...,Government Procurement Supplier,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
1,What topics does the Ministry of Finance's gui...,[### Visual Data\n\n#### Table of Contents\n| ...,The guide published by the Ministry of Finance...,Government Procurement Supplier,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What is the purpose of the guide for suppliers...,[### Key Text Segments\n\n- **Title**: About t...,The purpose of the guide is to assist supplier...,Public Procurement Officer,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,How does the WTO-GPA influence Singapore's gov...,[### Key Text Segments\n\n- **Title**: Alignme...,The WTO-GPA sets legal ground-rules for intern...,Government Procurement Supplier,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,What are the benefits of registering as a GeBI...,[<1-hop>\n\n### Key Text Segments\n- **Title:*...,Registering as a GeBIZ Trading Partner provide...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,What are the key government agencies involved ...,[<1-hop>\n\n### Visual Data\n\n#### Key Text S...,The key government agencies involved in procur...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,What are the key government agencies involved ...,[<1-hop>\n\n### Visual Data\n\n#### Key Text S...,The key government agencies involved in procur...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,What are the key terms and conditions related ...,[<1-hop>\n\n### Visual Data\n\n#### Table: Pro...,The key terms and conditions related to the In...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,What is the role of the Building and Construct...,[<1-hop>\n\n### Visual Data\n\n#### Key Text S...,The Building and Construction Authority (BCA) ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,What role does the Ministry of Trade & Industr...,[<1-hop>\n\n### Key Text Segments\n- **Title:*...,The Ministry of Trade & Industry (MTI) is invo...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [83]:
df.iloc[9]["user_input"]

'What role does the Ministry of Trade & Industry (MTI) play in providing feedback on government procurement processes?'

In [82]:
print(df.iloc[9]["reference_contexts"][0])

<1-hop>

### Key Text Segments
- **Title:** 9. Providing Feedback to the Government
- **Introduction:** The Government endeavours to uphold high standards in its procurement process and will continuously review its process as necessary. Your feedback is important to highlight examples of both good practices and areas for improvement.

### Visual Data

#### Flowchart
- **The Procuring Agency**
  - As a start, get in touch with the agency who is calling the procurement for clarifications.
  - The agency is obliged to explain to you why your bid was not successful, upon writing in.

- **The Ministry of Finance**
  - The Ministry of Finance (MOF) is in charge of the procurement policy for the entire Government. 
  - If you have any concerns over a Quotation or Tender, or feedback on how to improve our procurement practices and processes, you may direct them to MOF via mof_qsm@mof.gov.sg.
  - If you have experiences to share on accessing overseas markets or doing business with foreign Gover

In [84]:
df.iloc[9]["reference"]

'The Ministry of Trade & Industry (MTI) is involved in the feedback process regarding government procurement practices. Suppliers can direct their experiences and feedback on accessing overseas markets or doing business with foreign governments to the Ministry of Finance (MOF), which collaborates with MTI. This highlights the importance of MTI in addressing concerns and improving procurement practices.'